In [0]:
# Databricks Notebook: Customer_360
# Cell 1: Aggregate Customer Financial Profile

from pyspark.sql.functions import col, coalesce, count, current_timestamp, lit, sum

# 1. Read Silver Layer Entities
df_customer = spark.table("bankingpoc.silver.customer").filter(col("is_current") == True)
df_account = spark.table("bankingpoc.silver.account")
df_loan = spark.table("bankingpoc.silver.loan")
df_txn = spark.table("bankingpoc.silver.transaction")

# 2. Account Aggregations per Customer
df_acc_summary = df_account.groupBy("customer_id").agg(
    count("account_id").alias("total_accounts"),
    sum("balance").alias("total_deposit_balance")
)

# 3. Loan Aggregations per Customer
df_loan_summary = df_loan.groupBy("customer_id").agg(
    count("loan_id").alias("total_loans"),
    sum("loan_amount").alias("total_loan_amount"),
    sum("paid_amount").alias("total_loan_paid_amount")
)

# 4. Transaction Aggregations per Account joined to Customer
df_txn_summary = df_txn.join(df_account, "account_id", "inner") \
    .groupBy("customer_id") \
    .agg(
        count("transaction_id").alias("total_transactions"),
        sum(col("amount")).alias("total_transaction_volume")
    )

# 5. Join Dimensions and Aggregations
df_customer_360 = df_customer \
    .join(df_acc_summary, "customer_id", "left") \
    .join(df_loan_summary, "customer_id", "left") \
    .join(df_txn_summary, "customer_id", "left") \
    .select(
        col("customer_id"),
        col("first_name"),
        col("last_name"),
        col("email"),
        col("phone"),
        col("kyc_status"),
        coalesce(col("total_accounts"), lit(0)).alias("total_accounts"),
        coalesce(col("total_deposit_balance"), lit(0.00)).cast("decimal(18,2)").alias("total_deposit_balance"),
        coalesce(col("total_loans"), lit(0)).alias("total_loans"),
        coalesce(col("total_loan_amount"), lit(0.00)).cast("decimal(18,2)").alias("total_loan_amount"),
        coalesce(col("total_loan_paid_amount"), lit(0.00)).cast("decimal(18,2)").alias("total_loan_paid_amount"),
        coalesce(col("total_transactions"), lit(0)).alias("total_transactions"),
        coalesce(col("total_transaction_volume"), lit(0.00)).cast("decimal(18,2)").alias("total_transaction_volume"),
        current_timestamp().alias("gold_processed_timestamp")
    )

# 6. Overwrite Gold Delta Table
(df_customer_360.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bankingpoc.gold.customer_360"))

print("Gold table bankingpoc.gold.customer_360 successfully generated.")

Gold table bankingpoc.gold.customer_360 successfully generated.
